# 🔧 Feature Engineering

**목표**: 모든 가능한 Feature를 생성하고 그룹별로 관리

**전략**:
- EDA에서 발견한 인사이트 기반 Feature 생성
- 모든 Feature를 `full/` 폴더에 저장
- Feature 그룹을 JSON으로 관리 → 모델링에서 자유롭게 조합

---

## 1. Setup

In [1]:
# Libraries
import pandas as pd
import numpy as np
import os
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded")

✅ Libraries loaded


In [2]:
# Google Drive 연결
from google.colab import drive
drive.mount('/content/drive')

# 프로젝트 경로
PROJECT_PATH = '/content/drive/My Drive/Projects/smart-factory-power'
os.chdir(PROJECT_PATH)
print(f"✅ Working directory: {os.getcwd()}")

Mounted at /content/drive
✅ Working directory: /content/drive/My Drive/Projects/smart-factory-power


## 2. 데이터 로드

In [3]:
# 1시간 집계된 데이터 로드
print("📂 Loading hourly data...")

train = pd.read_csv('data/processed/hourly/train.csv')
test = pd.read_csv('data/processed/hourly/test.csv')

# datetime 변환
train['datetime'] = pd.to_datetime(train['datetime'])
test['datetime'] = pd.to_datetime(test['datetime'])

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"\nTrain period: {train['datetime'].min()} ~ {train['datetime'].max()}")
print(f"Test period: {test['datetime'].min()} ~ {test['datetime'].max()}")

📂 Loading hourly data...
Train shape: (2896, 25)
Test shape: (703, 25)

Train period: 2024-12-01 08:00:00 ~ 2025-03-31 23:00:00
Test period: 2025-04-01 00:00:00 ~ 2025-04-30 06:00:00


In [4]:
# 데이터 확인
print("\n📊 Train 데이터 샘플:")
display(train.head())

print("\n📋 컬럼 정보:")
print(train.columns.tolist())


📊 Train 데이터 샘플:


,datetime,activePower_mean,activePower_std,activePower_min,activePower_max,activePower_count,voltageR,voltageS,voltageT,voltageRS,...,powerFactorR,powerFactorS,powerFactorT,reactivePowerLagging,operation,hour,day,month,weekday,is_weekend
0,2024-12-01 08:00:00,3011.903014,738.202090,1051.34,5013.27,720,214.942208,215.099639,215.087583,372.416236,...,92.557458,92.273306,92.413792,597.735431,1.0,8,1,12,6,1
1,2024-12-01 09:00:00,3011.384833,736.284029,1126.53,5096.31,720,215.008472,215.164583,214.659667,372.529806,...,92.351417,92.409097,92.837611,611.235931,1.0,9,1,12,6,1
2,2024-12-01 10:00:00,2990.481861,710.641426,1046.87,4971.15,720,214.989569,215.172931,215.118292,372.520556,...,92.177236,92.469208,92.522000,592.212528,1.0,10,1,12,6,1
3,2024-12-01 11:00:00,3003.309125,721.679595,1158.87,4855.11,720,214.940111,215.070264,215.064778,372.388889,...,92.310389,92.590153,92.689306,605.942375,1.0,11,1,12,6,1
4,2024-12-01 12:00:00,3007.081417,709.555132,1109.05,4913.54,720,214.757653,215.074361,214.905472,372.234403,...,92.282694,92.257389,92.290847,611.667486,1.0,12,1,12,6,1



📋 컬럼 정보:
['datetime', 'activePower_mean', 'activePower_std', 'activePower_min', 'activePower_max', 'activePower_count', 'voltageR', 'voltageS', 'voltageT', 'voltageRS', 'voltageST', 'voltageTR', 'currentR', 'currentS', 'currentT', 'powerFactorR', 'powerFactorS', 'powerFactorT', 'reactivePowerLagging', 'operation', 'hour', 'day', 'month', 'weekday', 'is_weekend']


## 3. Feature Engineering Functions

### 3.1 시간 Features

In [5]:
def create_time_features(df):
    """
    시간 기반 Feature 생성
    - 영향력은 낮지만 (EDA 결과) 보조적으로 사용
    - Cyclic encoding (sin/cos)으로 주기성 표현
    - hour, day, month, weekday, is_weekend는 이미 존재
    """
    df = df.copy()


    # ✅ Cyclic encoding (주기성 표현)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['weekday_sin'] = np.sin(2 * np.pi * df['weekday'] / 7)
    df['weekday_cos'] = np.cos(2 * np.pi * df['weekday'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    return df

print("✅ create_time_features() defined")

✅ create_time_features() defined


### 3.2 Lag Features

In [6]:
def create_lag_features(df, target_col='activePower_mean', lags=[1, 2, 3]):
    """
    Lag Feature 생성
    - EDA에서 Lag 1 ACF = -0.49 발견!
    - 반전 패턴: 올라가면 내려가고, 내려가면 올라감
    - Lag 1, 2, 3이 가장 중요
    """
    df = df.copy()

    for lag in lags:
        df[f'{target_col}_lag_{lag}h'] = df[target_col].shift(lag)

    return df

print("✅ create_lag_features() defined")

✅ create_lag_features() defined


### 3.3 Rolling Features (변동성 캡처)

In [7]:
def create_rolling_features(df, target_col='activePower_mean', windows=[6, 12, 24]):
    """
    Rolling 통계 Feature 생성
    - EDA: 시간 내 변동이 시간 간 변동보다 27배 큼
    - 변동성을 캡처하는 것이 중요!
    """
    df = df.copy()

    for window in windows:
        # Rolling mean (추세)
        df[f'{target_col}_rolling_mean_{window}h'] = (
            df[target_col].rolling(window=window, min_periods=1).mean()
        )

        # Rolling std (변동성)
        df[f'{target_col}_rolling_std_{window}h'] = (
            df[target_col].rolling(window=window, min_periods=1).std()
        )

        # Rolling min/max (범위)
        df[f'{target_col}_rolling_min_{window}h'] = (
            df[target_col].rolling(window=window, min_periods=1).min()
        )
        df[f'{target_col}_rolling_max_{window}h'] = (
            df[target_col].rolling(window=window, min_periods=1).max()
        )

    return df

print("✅ create_rolling_features() defined")

✅ create_rolling_features() defined


### 3.4 차분 Features

In [8]:
def create_diff_features(df, target_col='activePower_mean', periods=[1, 2]):
    """
    차분 Feature 생성
    - 변화량을 직접적으로 표현
    - Lag와 함께 사용하면 효과적
    """
    df = df.copy()

    for period in periods:
        df[f'{target_col}_diff_{period}h'] = df[target_col].diff(period)

    return df

print("✅ create_diff_features() defined")

✅ create_diff_features() defined


### 3.5 통합 Feature 생성 함수

In [9]:
def create_all_features(df):
    """
    모든 Feature 생성 파이프라인
    """
    print("🔧 Creating all features...")

    df = df.copy()

    # 1. 시간 Features
    print("  ⏰ Time features...")
    df = create_time_features(df)

    # 2. Lag Features (최우선!)
    print("  🔙 Lag features...")
    df = create_lag_features(df, lags=[1, 2, 3])

    # 3. Rolling Features
    print("  📊 Rolling features...")
    df = create_rolling_features(df, windows=[6, 12, 24])

    # 4. 차분 Features
    print("  📉 Diff features...")
    df = create_diff_features(df, periods=[1, 2])


    print(f"\n✅ Total features created: {df.shape[1]}")

    return df

print("✅ create_all_features() defined")

✅ create_all_features() defined


## 4. Feature 생성 실행

In [10]:
# Train 데이터 Feature 생성
print("🚀 Creating features for TRAIN data...\n")
train_featured = create_all_features(train)

print("\n" + "="*50)
print("📊 Train Featured Data Info:")
print(f"Shape: {train_featured.shape}")
print(f"Columns: {train_featured.columns.tolist()[:10]}...")

🚀 Creating features for TRAIN data...

🔧 Creating all features...
  ⏰ Time features...
  🔙 Lag features...
  📊 Rolling features...
  📉 Diff features...

✅ Total features created: 48

📊 Train Featured Data Info:
Shape: (2896, 48)
Columns: ['datetime', 'activePower_mean', 'activePower_std', 'activePower_min', 'activePower_max', 'activePower_count', 'voltageR', 'voltageS', 'voltageT', 'voltageRS']...


In [11]:
# Test 데이터 Feature 생성
print("🚀 Creating features for TEST data...\n")
test_featured = create_all_features(test)

print("\n" + "="*50)
print("📊 Test Featured Data Info:")
print(f"Shape: {test_featured.shape}")
print(f"Columns: {test_featured.columns.tolist()[:10]}...")

🚀 Creating features for TEST data...

🔧 Creating all features...
  ⏰ Time features...
  🔙 Lag features...
  📊 Rolling features...
  📉 Diff features...

✅ Total features created: 48

📊 Test Featured Data Info:
Shape: (703, 48)
Columns: ['datetime', 'activePower_mean', 'activePower_std', 'activePower_min', 'activePower_max', 'activePower_count', 'voltageR', 'voltageS', 'voltageT', 'voltageRS']...


## 5. Feature 검증

In [12]:
# 결측치 확인
print("🔍 결측치 확인:")
missing = train_featured.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) > 0:
    print(f"\n⚠️ 결측치가 있는 컬럼: {len(missing)}개")
    display(missing.head(20))
else:
    print("\n✅ 결측치 없음!")

🔍 결측치 확인:

⚠️ 결측치가 있는 컬럼: 8개


,0
activePower_mean_lag_3h,3
activePower_mean_lag_2h,2
activePower_mean_diff_2h,2
activePower_mean_lag_1h,1
activePower_mean_rolling_std_6h,1
activePower_mean_rolling_std_12h,1
activePower_mean_rolling_std_24h,1
activePower_mean_diff_1h,1


In [13]:
# Lag Feature로 인한 초기 결측치는 정상
print("\n📋 처음 5행 확인 (Lag로 인한 결측치):")
display(train_featured[[
    'datetime',
    'activePower_mean',
    'activePower_mean_lag_1h',
    'activePower_mean_lag_2h',
    'activePower_mean_lag_3h'
]].head())


📋 처음 5행 확인 (Lag로 인한 결측치):


,datetime,activePower_mean,activePower_mean_lag_1h,activePower_mean_lag_2h,activePower_mean_lag_3h
0,2024-12-01 08:00:00,3011.903014,NaN,NaN,NaN
1,2024-12-01 09:00:00,3011.384833,3011.903014,NaN,NaN
2,2024-12-01 10:00:00,2990.481861,3011.384833,3011.903014,NaN
3,2024-12-01 11:00:00,3003.309125,2990.481861,3011.384833,3011.903014
4,2024-12-01 12:00:00,3007.081417,3003.309125,2990.481861,3011.384833


In [14]:
# 무한대 값 확인
print("\n🔍 무한대 값 확인:")
inf_cols = []
for col in train_featured.select_dtypes(include=[np.number]).columns:
    if np.isinf(train_featured[col]).any():
        inf_cols.append(col)

if len(inf_cols) > 0:
    print(f"\n⚠️ 무한대 값이 있는 컬럼: {inf_cols}")
else:
    print("\n✅ 무한대 값 없음!")


🔍 무한대 값 확인:

✅ 무한대 값 없음!


In [15]:
# 주요 Feature 통계 확인
print("\n📊 주요 Feature 통계:")
key_features = [
    'activePower_mean',
    'activePower_mean_lag_1h',
    'activePower_mean_rolling_mean_6h',
    'activePower_mean_rolling_std_24h'
]
display(train_featured[key_features].describe())


📊 주요 Feature 통계:


,activePower_mean,activePower_mean_lag_1h,activePower_mean_rolling_mean_6h,activePower_mean_rolling_std_24h
count,2896.000000,2895.000000,2896.000000,2895.000000
mean,3010.248489,3010.258992,3010.249782,26.479823
std,26.730057,26.728697,11.255596,3.578366
min,2927.116861,2927.116861,2971.510155,0.366409
25%,2991.962351,2991.995465,3002.564746,24.059748
50%,3010.078333,3010.079903,3010.269647,26.383295
75%,3028.349538,3028.361771,3017.988694,28.968623
max,3106.273153,3106.273153,3054.589414,39.168700


## 6. Feature 그룹 정의

In [16]:
# 원본 컬럼 (Feature 아님)
original_cols = [
    'datetime',
    'activePower_mean', 'activePower_std', 'activePower_min', 'activePower_max', 'activePower_count',
    'voltageR', 'voltageS', 'voltageT',
    'voltageRS', 'voltageST', 'voltageTR',
    'currentR', 'currentS', 'currentT',
    'powerFactorR', 'powerFactorS', 'powerFactorT',
    'reactivePowerLagging', 'operation',
    'hour', 'day', 'month', 'weekday', 'is_weekend'
]

# Feature 컬럼만 추출
all_features = [col for col in train_featured.columns if col not in original_cols]

print(f"✅ 원본 컬럼: {len(original_cols)}개")
print(f"✅ Feature 컬럼: {len(all_features)}개")
print(f"✅ 전체 컬럼: {train_featured.shape[1]}개")

✅ 원본 컬럼: 25개
✅ Feature 컬럼: 23개
✅ 전체 컬럼: 48개


In [17]:
feature_groups = {
    # 시간 Features는 이미 원본에 있으므로 cyclic만
    "time_cyclic": [
        'hour_sin', 'hour_cos',
        'weekday_sin', 'weekday_cos',
        'month_sin', 'month_cos'
    ],

    # Lag Features (최우선!)
    "lag": [
        'activePower_mean_lag_1h',
        'activePower_mean_lag_2h',
        'activePower_mean_lag_3h'
    ],

    # Rolling Features
    "rolling_6h": [
        'activePower_mean_rolling_mean_6h',
        'activePower_mean_rolling_std_6h',
        'activePower_mean_rolling_min_6h',
        'activePower_mean_rolling_max_6h'
    ],

    "rolling_12h": [
        'activePower_mean_rolling_mean_12h',
        'activePower_mean_rolling_std_12h',
        'activePower_mean_rolling_min_12h',
        'activePower_mean_rolling_max_12h'
    ],

    "rolling_24h": [
        'activePower_mean_rolling_mean_24h',
        'activePower_mean_rolling_std_24h',
        'activePower_mean_rolling_min_24h',
        'activePower_mean_rolling_max_24h'
    ],

    # 차분 Features
    "diff": [
        'activePower_mean_diff_1h',
        'activePower_mean_diff_2h'
    ]
}

## 7. 저장

In [18]:
# 저장 디렉토리 생성
output_dir = 'data/processed/featured/full'
os.makedirs(output_dir, exist_ok=True)

print(f"📁 Output directory: {output_dir}")

📁 Output directory: data/processed/featured/full


In [19]:
# CSV 저장
print("💾 Saving featured data...")

train_featured.to_csv(f'{output_dir}/train.csv', index=False, encoding='utf-8-sig')
test_featured.to_csv(f'{output_dir}/test.csv', index=False, encoding='utf-8-sig')

print(f"  ✅ train.csv: {train_featured.shape}")
print(f"  ✅ test.csv: {test_featured.shape}")

💾 Saving featured data...
  ✅ train.csv: (2896, 48)
  ✅ test.csv: (703, 48)


In [20]:
# Feature 그룹 JSON 저장
print("\n💾 Saving feature groups...")

with open(f'{output_dir}/feature_groups.json', 'w', encoding='utf-8') as f:
    json.dump(feature_groups, f, indent=2, ensure_ascii=False)

print(f"  ✅ feature_groups.json saved")


💾 Saving feature groups...
  ✅ feature_groups.json saved


In [21]:
# Feature 목록 TXT 저장
print("\n💾 Saving feature list...")

with open(f'{output_dir}/features.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(all_features))

print(f"  ✅ features.txt: {len(all_features)}개 저장")


💾 Saving feature list...
  ✅ features.txt: 23개 저장


In [22]:
# 메타데이터 저장
print("\n💾 Saving metadata...")

metadata = {
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "train_shape": train_featured.shape,
    "test_shape": test_featured.shape,
    "n_features": len(all_features),
    "n_groups": len(feature_groups),
    "groups": {k: len(v) for k, v in feature_groups.items()}
}

with open(f'{output_dir}/metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"  ✅ metadata.json saved")


💾 Saving metadata...
  ✅ metadata.json saved


## 8. 최종 확인

In [23]:
# 저장된 파일 확인
print("📂 저장된 파일 목록:")
for file in os.listdir(output_dir):
    file_path = os.path.join(output_dir, file)
    file_size = os.path.getsize(file_path) / 1024 / 1024  # MB
    print(f"  ✅ {file:25s} ({file_size:.2f} MB)")

📂 저장된 파일 목록:
  ✅ features.txt              (0.00 MB)
  ✅ test.csv                  (0.49 MB)
  ✅ metadata.json             (0.00 MB)
  ✅ feature_groups.json       (0.00 MB)
  ✅ train.csv                 (2.02 MB)


In [24]:
# 샘플 데이터 확인
print("\n📊 저장된 데이터 샘플:")
display(train_featured.head())

print("\n📊 Feature 통계:")
display(train_featured[all_features[:5]].describe())  # 처음 5개 Feature만


📊 저장된 데이터 샘플:


,datetime,activePower_mean,activePower_std,activePower_min,activePower_max,activePower_count,voltageR,voltageS,voltageT,voltageRS,...,activePower_mean_rolling_mean_12h,activePower_mean_rolling_std_12h,activePower_mean_rolling_min_12h,activePower_mean_rolling_max_12h,activePower_mean_rolling_mean_24h,activePower_mean_rolling_std_24h,activePower_mean_rolling_min_24h,activePower_mean_rolling_max_24h,activePower_mean_diff_1h,activePower_mean_diff_2h
0,2024-12-01 08:00:00,3011.903014,738.202090,1051.34,5013.27,720,214.942208,215.099639,215.087583,372.416236,...,3011.903014,NaN,3011.903014,3011.903014,3011.903014,NaN,3011.903014,3011.903014,NaN,NaN
1,2024-12-01 09:00:00,3011.384833,736.284029,1126.53,5096.31,720,215.008472,215.164583,214.659667,372.529806,...,3011.643924,0.366409,3011.384833,3011.903014,3011.643924,0.366409,3011.384833,3011.903014,-0.518181,NaN
2,2024-12-01 10:00:00,2990.481861,710.641426,1046.87,4971.15,720,214.989569,215.172931,215.118292,372.520556,...,3004.589903,12.220669,2990.481861,3011.903014,3004.589903,12.220669,2990.481861,3011.903014,-20.902972,-21.421153
3,2024-12-01 11:00:00,3003.309125,721.679595,1158.87,4855.11,720,214.940111,215.070264,215.064778,372.388889,...,3004.269708,9.998663,2990.481861,3011.903014,3004.269708,9.998663,2990.481861,3011.903014,12.827264,-8.075708
4,2024-12-01 12:00:00,3007.081417,709.555132,1109.05,4913.54,720,214.757653,215.074361,214.905472,372.234403,...,3004.832050,8.749920,2990.481861,3011.903014,3004.832050,8.749920,2990.481861,3011.903014,3.772292,16.599556



📊 Feature 통계:


,hour_sin,hour_cos,weekday_sin,weekday_cos,month_sin
count,2896.000000,2.896000e+03,2896.000000,2896.000000,2.896000e+03
mean,-0.001818,-1.394702e-03,-0.004320,0.011732,5.863153e-01
std,0.707121,7.073333e-01,0.704715,0.709623,3.891689e-01
min,-1.000000,-1.000000e+00,-0.974928,-0.900969,-2.449294e-16
25%,-0.707107,-7.071068e-01,-0.781831,-0.900969,-2.449294e-16
50%,0.000000,-1.836970e-16,0.000000,-0.222521,5.000000e-01
75%,0.707107,7.071068e-01,0.781831,0.623490,1.000000e+00
max,1.000000,1.000000e+00,0.974928,1.000000,1.000000e+00
